# Search SWOT
https://github.com/SWOT-community/SWOT-Oceanography/blob/main/ex_search_swot.ipynb

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import geopandas as gpd
from shapely import geometry

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
os.chdir('/g/data/jk72/jl1058/SWOT_processing')

In [ ]:
# def get_half_orbits_intersect(bbox):
#     """Get half orbits that intersect a bounding box.

#     Parameters
#     ----------
#     bbox: 
#         the bounding box
        
#     Returns
#     -------
#      gpd.GeoDataFrame:
#         A Geopandas dataframe containing intersecting half orbits numbers and geometries
#     """
#     swath_geometries = gpd.read_file(GEOMETRIES_FILE)

#     bbox_polygon = geometry.box(*bbox)

#     def _filter_intersect(row, polygon):
#         half_orbit_polygon = row.geometry
#         return polygon.intersects(half_orbit_polygon)

#     select = swath_geometries.apply(_filter_intersect, polygon=bbox_polygon, axis=1)
#     return swath_geometries[select]


In [ ]:
def get_half_orbits_intersect(bbox):
    """Get half orbits that intersect a bounding box, including intersection area.

    Parameters
    ----------
    bbox: tuple or list
        (minx, miny, maxx, maxy)

    Returns
    -------
    gpd.GeoDataFrame:
        A GeoDataFrame containing intersecting half-orbits, their geometries,
        and the area of intersection with the bbox.
    """
    swath_geometries = gpd.read_file(GEOMETRIES_FILE)

    bbox_polygon = geometry.box(*bbox)

    def _compute_intersection_area(row, polygon):
        half_orbit_polygon = row.geometry
        if half_orbit_polygon.intersects(polygon):
            inter = half_orbit_polygon.intersection(polygon)
            return inter.area
        return 0.0

    # Compute intersection areas
    swath_geometries["intersection_area"] = swath_geometries.apply(
        _compute_intersection_area,
        polygon=bbox_polygon,
        axis=1
    )

    # Select only intersecting ones
    return swath_geometries[swath_geometries["intersection_area"] > 0]


In [ ]:
# --------------------------------------------------------------------------- # 

def plot_geometries(geometries, lon_range, lat_range, title, plot_extent=None):
    fig, ax = plt.subplots(ncols=1, figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree()})

    gpd.GeoSeries(geometries.geometry).plot(ax=ax,transform=ccrs.PlateCarree(),alpha=1)
    
    square = patches.Rectangle((lon_range[0], lat_range[0]), lon_range[1]-lon_range[0], lat_range[1]-lat_range[0], edgecolor='orange', facecolor='none', transform=ccrs.PlateCarree())
    ax.add_patch(square)

    ax.set_title(title)
    ax.coastlines()
    if plot_extent:
        ax.set_extent(plot_extent, crs=ccrs.PlateCarree())


In [ ]:
def delete_unmatched_pass_files(directory, filtered, dry_run=True):
    """
    Print (dry run) or delete files whose pass number is not in the filtered swath_geometries.
    
    Parameters
    ----------
    directory : str
        Path to directory containing SWOT .nc files
    filtered : GeoDataFrame
        Filtered GeoDataFrame containing valid half-orbit pass numbers.
        Must include a column "pass_number" (int or str)
    dry_run : bool
        If True, only print what would be deleted. If False, delete the files.
    """

    # Convert all pass numbers in filtered to integers
    valid_passes = set(filtered["pass_number"].astype(int))

    for filename in os.listdir(directory):
        if not filename.endswith(".nc"):
            continue

        parts = filename.split('_')
        if len(parts) < 7:
            continue  # skip malformed filenames

        # Extract pass number from filename and convert to int
        try:
            pass_num = int(parts[6])  # 6th index (7th field) is the pass number
        except ValueError:
            continue  # skip if not an integer

        # Delete if pass number is NOT in filtered
        if pass_num not in valid_passes:
            file_path = os.path.join(directory, filename)
            if dry_run:
                print(f"[DRY RUN] Would delete: {file_path}")
            else:
                print(f"Deleting: {file_path}")
                os.remove(file_path)


In [ ]:
# Mawson Coast
# lon_range = 50, 75
# lat_range = -70, -60

# Kapp Norvegia
# lon_range = -18, -3
# lat_range = -73, -68

# Svalbard
lon_range = 8, 28
lat_range = 76, 82

# Thwaites
lon_range = -113, -104
lat_range = -72, -75.5



bbox = [lon_range[0], lat_range[0], lon_range[1], lat_range[1]]
plot_extent = [lon_range[0]-20, lon_range[1]+20, lat_range[0]-5, lat_range[1]+5]

In [ ]:
# science, calval
phase = 'science'

In [ ]:
GEOMETRIES_FILE = f'KaRIn_2kms_{phase}_geometries.geojson'

In [ ]:
swath_geoms = get_half_orbits_intersect(bbox)

In [ ]:
# swath_geoms.iloc[[0]]
swath_geoms

In [ ]:
# swath_geometries is your GeoDataFrame with an 'intersection_area' column
values = swath_geoms["intersection_area"].dropna()

plt.figure(figsize=(8, 5))
plt.hist(values, bins=30)   # No color/style specified (per requirements)
plt.xlabel("Intersection Area")
plt.ylabel("Frequency")
plt.title("Histogram of Intersection Area")
plt.show()

In [ ]:
# area_threshold = 20 # Mawson Coast, Svalbard
# area_threshold = 15 # Kapp Norvegia
area_threshold = 0 # Thwaites

swath_geoms_filt = swath_geoms[swath_geoms["intersection_area"] > area_threshold]
len(swath_geoms_filt)

In [ ]:
half_orbits = list(swath_geoms_filt['pass_number'])
half_orbits

In [ ]:
# plot_geometries(swath_geoms.iloc[[0]], lon_range, lat_range, f"Swot {phase} Geometries", plot_extent)
plot_geometries(swath_geoms_filt, lon_range, lat_range, f"Swot {phase} Geometries", plot_extent)


In [ ]:
# # In case existing files in dir not matching filetred pass numbers need to be deleted
# L3_outdir = "/g/data/gv90/jw2777/SWOT_SEAICE/DATA/SWOT_L3_Unsmoothed_Full/"
# delete_unmatched_pass_files(L3_outdir, swath_geoms_filt, dry_run=True) # turn dry_run to False


# SWOT DL
https://github.com/SWOT-community/SWOT-Oceanography/blob/main/ex_download_subset_aviso_ftp.ipynb

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
from getpass import getpass
import ftplib
from urllib.parse import urlparse
import xarray as xr
from concurrent.futures import ThreadPoolExecutor, as_completed


In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

In [ ]:
os.chdir('/g/data/jk72/jl1058/SWOT_processing')
# import swot_helpers

In [ ]:
import re
import time

def _download_file_thread(ftp_details, ftp_path, filename, target_directory, max_retries=3):
    """Download a single file with a new FTP connection (thread-safe),
    skip if a file with same _CCC_HHH_ exists, retry on failure.
    """
    ftp_host, username, password = ftp_details
    local_filepath = os.path.join(target_directory, filename)

    # ------------------------------------------------------------------
    # Extract the cycle_half_orbit key, e.g. "_016_551_"
    match = re.search(r'_(\d{3})_(\d{3})_', filename)
    if match:
        key = f"_{match.group(1)}_{match.group(2)}_"
    else:
        key = None
    # ------------------------------------------------------------------

    # Skip if any file in target_directory contains the same key
    if key is not None:
        for existing_file in os.listdir(target_directory):
            if key in existing_file:
                print(f"Skipping {filename}, matching file already exists: {existing_file}")
                return os.path.join(target_directory, existing_file)

    # ------------------------------------------------------------------
    # Download logic
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Downloading {filename}, attempt {attempt}")
            with ftplib.FTP(ftp_host) as ftp:
                ftp.login(username, password)
                ftp.cwd(ftp_path)
                with open(local_filepath, 'wb') as file:
                    ftp.retrbinary(f'RETR {filename}', file.write)
            print(f"Downloaded {filename}")
            return local_filepath

        except ftplib.error_perm as e:
            if '530' in str(e):
                print(f"FTP session limit reached, waiting before retrying {filename}...")
                time.sleep(10)
            else:
                print(f"Permission error downloading {filename}: {e}")
                break

        except Exception as e:
            print(f"Error downloading {filename} (attempt {attempt}): {e}")
            time.sleep(5)

    print(f"Failed to download {filename} after {max_retries} attempts.")
    return None

# --------------------------------------------------------------------------- # 

def _get_last_version_filename(filenames):
    versions = {int(f[-5:-3]): f for f in filenames}
    return versions[max(versions.keys())]

# --------------------------------------------------------------------------- # 

def _select_filename(filenames, only_last):
    if not only_last: return filenames
    return [_get_last_version_filename(filenames)]

# --------------------------------------------------------------------------- # 

def ftp_download_files_parallel(ftp_path, level, variant, cycle_numbers, half_orbits, output_dir, only_last=True, max_workers=4):
    """Download half orbit files in parallel from AVISO's FTP server."""
    ftpAVISO = 'ftp-access.aviso.altimetry.fr'
    ftp_details = (ftpAVISO, username, password)  # Use your FTP credentials here

    downloaded_files = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []

        for cycle in cycle_numbers:
            cycle_str = f'{cycle:03d}'
            cycle_dir = f'cycle_{cycle_str}'
            full_cycle_path = os.path.join(ftp_path, cycle_dir)

            for half_orbit in half_orbits:
                half_orbit_str = f'{half_orbit:03d}'
                pattern = f'SWOT_{level}_LR_SSH_{variant}_{cycle_str}_{half_orbit_str}'

                # Create a temporary FTP connection to list files
                try:
                    with ftplib.FTP(ftpAVISO) as ftp:
                        ftp.login(username, password)
                        ftp.cwd(full_cycle_path)
                        filenames = ftp.nlst(f'{pattern}_*')
                        if level == "L3":
                            only_last_flag = False
                        else:
                            only_last_flag = only_last
                        filenames = _select_filename(filenames, only_last_flag)
                except Exception as e:
                    print(f"No pass {half_orbit} in cycle {cycle}: {e}")
                    filenames = []

                for f in filenames:
                    futures.append(
                        executor.submit(_download_file_thread, ftp_details, full_cycle_path, f, output_dir, 3)
                    )

        for future in as_completed(futures):
            result = future.result()
            if result:
                downloaded_files.append(result)

    return downloaded_files

# --------------------------------------------------------------------------- # 

def _normalized_ds(ds, lon_min, lon_max):
    lon = ds.longitude.values
    lon[lon < lon_min] += 360
    lon[lon > lon_max] -= 360
    ds.longitude.values = lon
    return ds

# --------------------------------------------------------------------------- # 

def _subset_ds(file, variables, lon_range, lat_range, output_dir):
    print(f"Subset dataset: {file}")
    swot_ds = xr.open_dataset(file)
    swot_ds = swot_ds[variables]
    swot_ds.load()
    
    ds = _normalized_ds(swot_ds.copy(), -180, 180)

    mask = (
        (ds.longitude <= lon_range[1])
        & (ds.longitude >= lon_range[0])
        & (ds.latitude <= lat_range[1])
        & (ds.latitude >= lat_range[0])
    ).compute()
    
    swot_ds_area = swot_ds.where(mask, drop=True)

    if swot_ds_area.sizes['num_lines'] == 0:
        print(f'Dataset {file} not matching geographical area.')
        return None

    for var in list(swot_ds_area.keys()):
        swot_ds_area[var].encoding = {'zlib':True, 'complevel':5}

    filename = f"subset_{os.path.basename(urlparse(file).path)}"
    
    print(f"Store subset: {filename}")
    filepath = os.path.join(output_dir, filename)
    swot_ds_area.to_netcdf(filepath, mode='w')
        
    return filepath

# --------------------------------------------------------------------------- # 

def subset_files_parallel(filenames, variables, lon_range, lat_range, output_dir, max_workers=4):
    """Subset datasets in parallel with a given geographical area."""
    subsetted_files = []

    # Submit tasks to the executor
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_subset_ds, f, variables, lon_range, lat_range, output_dir): f for f in filenames}

        for future in as_completed(futures):
            result = future.result()
            if result is not None:
                subsetted_files.append(result)

    return subsetted_files

# --------------------------------------------------------------------------- # 

def plot_datasets(filenames, variable, extent=None):
    cb_args = dict(
        add_colorbar=True,
        cbar_kwargs={"shrink": 0.3}
    )
    
    plot_kwargs = dict(
        x="longitude",
        y="latitude",
        cmap="Spectral_r",
        vmin=-0.2,
        vmax=0.2,
    )

    fig, ax = plt.subplots(figsize=(12, 8), subplot_kw=dict(projection=ccrs.PlateCarree()))
    if extent: ax.set_extent(extent)
    
    for filename in filenames:
        ds = xr.open_dataset(filename)
        ds[variable].plot.pcolormesh(
            ax=ax,
            **plot_kwargs,
            **cb_args)
        cb_args=dict(add_colorbar=False)
    
    ax.coastlines()
    ax.gridlines(draw_labels=True)




In [ ]:
from pathlib import Path

# Where you want the L3 Unsmoothed files to go
L3_OUT_DIR = Path("/g/data/gv90/jw2777/SWOT_SEAICE/DATA/SWOT_L3_Unsmoothed_Full/")
L3_OUT_DIR.mkdir(parents=True, exist_ok=True)

output_dir = L3_OUT_DIR

In [ ]:
# AVISO+ credentials (export these in your environment before running)
username = "jack.landy@bristol.ac.uk"
password = "REDACTED_USE_ENVIRONMENT"

In [ ]:
### SELECT VERSION ###
ftp_path = '/swot_products/l3_karin_nadir/l3_lr_ssh/v2_0_1/Unsmoothed/'
# ftp_path = '/swot_products/l3_karin_nadir/l3_lr_ssh/v1_0_2/Unsmoothed/'

# For selecting files with a regex pattern
# Basic, Expert, WindWave, Unsmoothed
variant = "Unsmoothed"
# L2, L3
level = "L3"

In [ ]:
# Define cycles and half_orbits numbers to download
# cycle_numbers = list(range(11, 30)) # Mawson Coast
cycle_numbers = list(range(1,43))
pass_numbers = half_orbits

print(cycle_numbers)

In [ ]:
downloaded_files = ftp_download_files_parallel(ftp_path, level, variant, cycle_numbers, pass_numbers, output_dir, only_last=True, max_workers=4)